In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import re
import seaborn as sns
from collections import OrderedDict
from scipy.spatial import ConvexHull
from tqdm.notebook import tqdm

from mx2che_tools import (
    compositionTranform,
    element_content,
    encodeRGB,
    formula2string, 
    getEhull,
    simplexBasis, 
)

In [ ]:
elements = ['W', 'Mo', 'U', 'Se', 'S', 'Np']
unique_elements = ['W', 'Mo', 'Se', 'S']

simplex_basis = simplexBasis(len(unique_elements) - 1)

In [ ]:
## Neat elements
neat_elements = pd.read_pickle("data/neat_elements_dataset.pkl.gz")

## Simplex basis df formation
simplex_basis_df = pd.DataFrame(simplex_basis, columns=[f'SC{i}' for i in range(len(simplex_basis) - 1)])
simplex_basis_df['formation_energy_pa'] = 0
simplex_basis_df['formula'] = neat_elements.loc[unique_elements]['formula'].values
simplex_basis_df['formula_str'] = simplex_basis_df['formula'].apply(formula2string)
simplex_basis_df['compound'] = unique_elements
simplex_basis_df['part'] = 'Neat elements'
simplex_basis_df['size'] = 6

## DFT-derived structures
data_dft = pd.read_pickle('data/ccs_dataset.pkl.gz')
print('Collecting DFT-derived structures...')
compositions = []
for i in tqdm(data_dft.index):
    compositions += [[element_content(element, data_dft.loc[i], relative=False) for element in unique_elements]]
compositions = np.array(compositions)

## DFT simplex coords formation
simplex_coords_dft = pd.DataFrame(list(map(lambda x: compositionTranform(x, simplex_basis), compositions)),
                                  columns=[f'SC{i}' for i in range(len(simplex_basis) - 1)])
simplex_coords_dft['formation_energy_pa'] = data_dft['formation_energy_per_atom'].values
simplex_coords_dft['formula'] = data_dft['formula'].values
simplex_coords_dft['formula_str'] = simplex_coords_dft['formula'].apply(formula2string)
simplex_coords_dft['compound'] = data_dft['compound'].values
simplex_coords_dft['part'] = data_dft['Space_group_no'].apply(lambda x: 'CCS (Pmm2 and higher)' if x > 8 else 'CCS (Cm and lower)').values
simplex_coords_dft['size'] = data_dft['Space_group_no'].apply(lambda x: (4 if x > 8 else 1)).values

## 2DMD dataset
data_2dmd = pd.read_pickle('data/2d-materials-point-defects-all-table.pkl.gz')
data_2dmd = data_2dmd[(data_2dmd['base'] == 'WSe2') | (data_2dmd['base'] == 'MoS2')].copy()
data_2dmd['natoms'] = data_2dmd['atomic_numbers'].apply(len)
data_2dmd['formation_energy_per_atom'] = data_2dmd['formation_energy'] / data_2dmd['natoms']
print('Collecting structures of the 2DMD dataset...')
compositions = []
for i in tqdm(data_2dmd.index):
    compositions += [[element_content(element, data_2dmd.loc[i], relative=False) for element in unique_elements]]
compositions = np.array(compositions)

## 2DMD simplex coords formation
simplex_coords_2dmd = pd.DataFrame(list(map(lambda x: compositionTranform(x, simplex_basis), compositions)),
                                   columns=[f'SC{i}' for i in range(len(simplex_basis) - 1)])
simplex_coords_2dmd['formation_energy_pa'] = data_2dmd['structure_formation_energy_per_atom'].values
simplex_coords_2dmd['formula'] = data_2dmd['formula'].values
simplex_coords_2dmd['formula_str'] = simplex_coords_2dmd['formula'].apply(formula2string)
simplex_coords_2dmd['compound'] = data_2dmd.index
simplex_coords_2dmd['part'] = '2DMD datasets'
simplex_coords_2dmd['size'] = .3

## Simplex df concatenation
simplex_df = pd.concat((simplex_basis_df, simplex_coords_dft, simplex_coords_2dmd), axis=0)
simplex_df.set_index('compound', inplace=True, drop=True)

In [ ]:
fig = px.scatter_3d(
    simplex_df, x='SC0', y='SC1', z='SC2', color='part', size='size', 
    color_discrete_sequence=px.colors.qualitative.Bold,
    hover_data=[
        simplex_df['part'], 
        simplex_df['formation_energy_pa'],
        simplex_df.index, 
        simplex_df['formula_str']
    ]
)
fig.update_traces(marker=dict(line=dict(width=0,)), selector=dict(mode='markers'))

In [ ]:
camera = {"eye": {"x": 1.5, "y": 0.5, "z": 0.9}}
fig.update_layout(scene_camera=camera)
fig.show()

In [ ]:
available_energies_indexes = simplex_df.index # all structures indexes
available_parts = simplex_df['part'].unique() # unique dataset parts

print(len(available_energies_indexes), 'energies available in ', ','.join(available_parts), 'parts')

# Convex hull object creation
points = simplex_df.iloc[:, :len(unique_elements)].values
hull = ConvexHull(points)
print(f'Convex hull dimensionality is {hull.ndim}D')

# Energies df formation
energies_df = pd.DataFrame(index=available_energies_indexes)
energies_df['Reference_hull_E'] = [getEhull(point, hull) for point in points[:, :3]]
energies_df['E_above_hull'] = simplex_df['formation_energy_pa'].values - energies_df['Reference_hull_E']
energies_df['E_above_hull'] = energies_df['E_above_hull'].apply(lambda x: x if np.abs(x) > 1e-05 else 0)

list_of_sets = list(map(set, hull.simplices))
list_of_hull_indexes = list(list_of_sets[0].union(*list_of_sets[1:]))
energies_df['DefineHull'] = np.isin(range(energies_df.shape[0]), list_of_hull_indexes) 
energies_df['DefineHull'] = (energies_df['DefineHull']) & (simplex_df['formation_energy_pa'].values < 1e-05) #negative energies only

energies_df = pd.concat((simplex_df, energies_df), axis=1)
energies_df['formula_str'] = energies_df['formula'].apply(formula2string)
energies_df['<25meV/atom_above'] = energies_df['E_above_hull'] < 25e-3
energies_df.info()

In [ ]:
result_energies_df = pd.merge(data_dft, energies_df, how='left', left_on='compound', right_on=energies_df.index)
space_groups = dict(zip(data_dft['Space_group_no'], data_dft['Space_group_symbol']))
space_groups = OrderedDict(sorted(space_groups.items(), reverse=True))
sns.pairplot(
    result_energies_df[[
        'W_nsites','U_nsites', 'Se_nsites', 'Np_nsites',
        'Space_group_symbol', 'formation_energy_per_atom', 'E_above_hull', 
        'defect_nsites'
    ]], 
    hue='Space_group_symbol', corner=True,
    hue_order=[item[1] for item in space_groups.items() if item[0] in data_dft['Space_group_no'].unique()], palette='crest'
)
plt.tight_layout()
plt.show()

In [ ]:
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=map(encodeRGB, px.colors.qualitative.Bold))
plt.figure(figsize=(6,4))
plt.hist(
    [1000 * energies_df[energies_df['part'] == part]['E_above_hull'] for part in available_parts], 
    bins=np.linspace(0, 1600, 65), stacked=True, label=available_parts, edgecolor='grey'
)
plt.xlabel('$E_{hull}$, meV/atom')
plt.ylabel('Number of structures')
plt.legend()
plt.grid(alpha=0.3)
plt.yscale('log')
plt.tight_layout()
plt.show()
plt.rcParams['axes.prop_cycle'] = plt.rcParamsDefault['axes.prop_cycle']

In [ ]:
missed_data = [col for col in energies_df.columns if not col in data_2dmd.columns]
data_2dmd_energies = pd.concat((data_2dmd, energies_df[energies_df['part'] == '2DMD datasets'][missed_data]), axis=1)
sns.pairplot(
    data_2dmd_energies[['defect_nsites', 'base', 'structure_formation_energy_per_atom', 'formation_energy_per_atom']], 
    height=3, hue='base', palette='tab10', corner=True, 
    plot_kws=dict(marker="o", linewidth=1, alpha=0.1)
)
plt.tight_layout()
plt.show()

print('Analysis of defect numbers remapping...')
print(data_2dmd['cell'].unique())
for dc in data_2dmd['defect_nsites'].unique():
    print(2 ** 2 * dc / 8 ** 2)

In [ ]:
sns.pairplot(
    data_2dmd_energies[[
        'homo_lumo_gap', 'defect_nsites', 'formation_energy_per_atom', 'base', 
        'space_group_no', 'structure_formation_energy_per_atom', 'E_above_hull'
    ]], 
    #height=3,
    hue='base', palette='tab10',corner=True, 
    plot_kws=dict(marker="o", linewidth=1, alpha=0.1)
)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 4))
dft_numbers = data_dft.groupby(['defect_nsites','Space_group_no']).size().to_frame()\
          .rename(mapper={0: 'Number_of_structures'}, axis=1).reset_index()\
          .pivot(columns='defect_nsites', index='Space_group_no', values='Number_of_structures')
dft_numbers.sort_index(ascending=False, inplace=True)
ccs_space_groups2add22dmd = dft_numbers.index

data_2d_numbers = data_2dmd.groupby(['defect_nsites','space_group_no']).size().to_frame()\
          .rename(mapper={0: 'Number_of_structures'}, axis=1).reset_index()\
          .pivot(columns='defect_nsites', index='space_group_no', values='Number_of_structures')
data_2d_numbers = pd.concat(
    (
        data_2d_numbers, 
        pd.DataFrame(index=[sg for sg in ccs_space_groups2add22dmd if sg not in data_2d_numbers.index])
    ), 
    axis=0
)
data_2d_numbers.sort_index(ascending=False, inplace=True)
data_2d_numbers = data_2d_numbers.T

ax = sns.heatmap(
    np.log10(data_2d_numbers),
    cmap='YlOrBr', annot=data_2d_numbers, linewidth=.1, 
    cbar_kws={
        'label': 'lg (number of inequivalent structures)', 
        'drawedges' : False, "shrink": 0.9
    },
    fmt='g', xticklabels=list(map(lambda x: f'{space_groups[x]}\n({x})', data_2d_numbers.columns)),
)
ax.set(xlabel="Space group", ylabel="Number of defect positions")
ax.xaxis.tick_top()
plt.show()

In [ ]:
fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(8, 4), sharey=True)
for base in data_2dmd_energies['base'].unique():
    mask = data_2dmd_energies['base'] == base
    sns.scatterplot(
        x=1000 * data_2dmd_energies[mask]['formation_energy_per_atom'],
        y=1000 * data_2dmd_energies[mask]['E_above_hull'], 
        alpha=0.2, label=re.sub('2', '$_2$', base), ax=axs[1]
    )
    sns.scatterplot(
        x=data_2dmd_energies[mask]['defect_nsites'], 
        y=1000 * data_2dmd_energies[mask]['E_above_hull'], 
        alpha=0.2, ax=axs[0]
    )
axs[0].plot([3.5, 3.5], [0, 300], 'k', alpha=0.3, lw=1)
axs[0].text(3.5 - .5, 300, 'LDC', verticalalignment='top', horizontalalignment='right')
axs[0].text(3.5 + .5, 300, 'HDC', verticalalignment='top', horizontalalignment='left')
axs[1].plot([0, 300], [0, 300], 'k', alpha=0.3, lw=1, label='perfect\nmatch')
for i in range(2):
    axs[i].grid(alpha=0.3)
axs[1].legend(labelspacing=0.3)
axs[0].set_xlabel('Number of defect positions (2DMD)')
axs[1].set_xlabel('$E_{DF}$, meV/atom (2DMD)')
axs[0].set_ylabel('$E_{hull}$, meV/atom')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(4.25, 4), sharey=True)
for base in data_2dmd_energies['base'].unique():
    mask = data_2dmd_energies['base'] == base
    sns.scatterplot(
        x=data_2dmd_energies[mask]['defect_nsites'], 
        y=1000 * data_2dmd_energies[mask]['E_above_hull'], 
        alpha=0.2, ax=ax, label=re.sub('2', '$_2$', base)
    )
ax.plot([3.5, 3.5], [0, 300], 'k', alpha=0.3, lw=1)
ax.text(3.5 - .5, 300, 'LDC', verticalalignment='top', horizontalalignment='right')
ax.text(3.5 + .5, 300, 'HDC', verticalalignment='top', horizontalalignment='left')
ax.grid(alpha=0.3)
ax.legend(title='Base monolayer', labelspacing=0.3)
ax.set_xlabel('Number of defect positions (out of 192 available)')
ax.set_ylabel('2DMD datasets: $E_{hull}$, meV/atom')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(4.25, 4), sharey=True)
for base in data_2dmd_energies['base'].unique():
    mask = data_2dmd_energies['base'] == base
    sns.scatterplot(
        x=1000 * data_2dmd_energies[mask]['formation_energy_per_atom'],
        y=1000 * data_2dmd_energies[mask]['E_above_hull'], 
        alpha=0.2, label=re.sub('2', '$_2$', base), ax=ax
    )
ax.plot([0, 300], [0, 300], 'k', alpha=0.3, lw=1, label='perfect\nmatch')
ax.grid(alpha=0.3)
ax.legend(labelspacing=0.3)
ax.set_xlabel('$E_{DF}$, meV/atom (2DMD)')
ax.set_ylabel('$E_{hull}$, meV/atom')
plt.tight_layout()
plt.show()

In [ ]:
df_energies_available = energies_df.dropna().copy()
df_energies_available['size'] =  (3 - df_energies_available['E_above_hull'])
fig = px.scatter_3d(
    df_energies_available, x='SC0', y='SC1', z='SC2', color='E_above_hull', 
    size='size', size_max=20, opacity=0.5,
    hover_data=[df_energies_available['part'], df_energies_available.index, df_energies_available['formula_str']], 
    color_continuous_scale='inferno'
)
fig.update_traces(marker=dict(line=dict(width=0)), selector=dict(mode='markers'))
fig.show()

In [ ]:
limit_Ehull = 25e-3
df_energies_on_hull = energies_df[(energies_df['E_above_hull'] < limit_Ehull) | (energies_df['part'] == '2DMD subsets')].copy()
df_energies_on_hull = df_energies_on_hull[df_energies_on_hull['E_above_hull'] < limit_Ehull]
df_energies_on_hull['size'] =  20 * (limit_Ehull - df_energies_on_hull['E_above_hull'])
fig = px.scatter_3d(
    df_energies_on_hull, x='SC0', y='SC1', z='SC2', color='E_above_hull',
    size='size', size_max = 20, opacity=1,
    hover_data=[
        df_energies_on_hull['part'], 
        df_energies_on_hull['formation_energy_pa'],
        df_energies_on_hull.index, 
        df_energies_on_hull['formula_str']
    ], 
    color_continuous_scale='inferno'
)
fig.update_traces(marker=dict(line=dict(width=0)), selector=dict(mode='markers'))
fig.show()

In [ ]:
camera = dict(up=dict(x=-1, y=0, z=0),
              eye=dict(x=0, y=0, z=1.7))
fig.update_layout(scene_camera=camera)
fig.show()